# Notebook 10 — Core Validation Storytelling (Present vs Prediction-Adjusted)

Purpose:
- validate the core thesis claim that prediction creates meaningful operational value
- compare current reactive scenario vs prediction-adjusted intervention scenario
- justify cycle presence using empirical cycle-linked deterioration patterns
- translate model outputs into resource and healthcare impact terms

## Visualization Concepts in This Notebook

1) **Cycle Presence Plot**: shows how escalation changes with cycle progression.
2) **Present vs Prediction Bar Plot**: expected escalations under reactive vs proactive strategy.
3) **Category Impact Plot**: where intervention has strongest marginal benefit by patient complexity.
4) **Sensitivity Heatmap**: stress-test of improvement under varying coverage and efficacy assumptions.
5) **Resource Translation Table**: expected admissions avoided and case-management load changes.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
TABLE_DIR = PROJECT_ROOT / 'Results' / 'tables' / 'notebook10_stage_m'
FIG_DIR = PROJECT_ROOT / 'Results' / 'figures' / 'notebook10_stage_m'
REPORT_DIR = PROJECT_ROOT / 'Results' / 'reports' / 'notebook10_stage_m'
META_DIR = PROJECT_ROOT / 'Data' / 'metadata'
for d in [TABLE_DIR, FIG_DIR, REPORT_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Notebook 10 workspace ready')

Notebook 10 workspace ready


In [2]:
# Load core artifacts from previous stages
panel_path = PROJECT_ROOT / 'Results' / 'tables' / 'notebook03_phase_f' / 'phase_f_closed_loop_panel.parquet'
rec_path = PROJECT_ROOT / 'Results' / 'tables' / 'notebook09_stage_l' / 'stage_l_patient_category_recommendations.csv'
rec_summary_path = PROJECT_ROOT / 'Results' / 'tables' / 'notebook09_stage_l' / 'stage_l_medication_recommendation_summary.csv'
h_metrics_path = PROJECT_ROOT / 'Results' / 'tables' / 'notebook05_stage_h' / 'stage_h_baseline_model_performance.csv'
l_metrics_path = PROJECT_ROOT / 'Results' / 'tables' / 'notebook09_stage_l' / 'stage_l_model_metrics.csv'

panel_cols = ['patient_id', 'day', 'cycle_id_stage_f', 'months_since_cycle_start', 'response_state_active', 'baseline_admission_event', 'stage_f_escalation_event', 'hazard_prob_stage_f']
panel = pd.read_parquet(panel_path, columns=panel_cols)
rec = pd.read_csv(rec_path)
rec_summary = pd.read_csv(rec_summary_path)
h_metrics = pd.read_csv(h_metrics_path)
l_metrics = pd.read_csv(l_metrics_path)

overall = {
    'n_rows': int(len(panel)),
    'n_patients': int(panel['patient_id'].nunique()),
    'escalation_rate': float(panel['stage_f_escalation_event'].mean()),
    'admission_rate': float(panel['baseline_admission_event'].mean())
}

print('Core panel summary:', overall)
print('Stage H metrics rows:', len(h_metrics), '| Stage L metrics rows:', len(l_metrics))
rec.head(3)

Core panel summary: {'n_rows': 4900000, 'n_patients': 100000, 'escalation_rate': 0.011253469387755103, 'admission_rate': 0.008729183673469388}
Stage H metrics rows: 1 | Stage L metrics rows: 2


,patient_id,mean_instability,mean_hazard,max_cycle,max_months,escalation_rate,dx_burden,patient_category,recommended_medication_strategy
0,P000000,0.273085,0.007144,1,44,0.020408,0.020408,moderate_complexity,fluoxetine
1,P000001,0.130166,0.006427,0,48,0.000000,0.020408,low_complexity,escitalopram
2,P000002,0.317238,0.007479,1,45,0.020408,0.040816,moderate_complexity,escitalopram


In [3]:
# Cycle-presence justification (is deterioration linked to cycle progression?)
cycle_stats = panel.groupby('cycle_id_stage_f', as_index=False).agg(
    n=('patient_id', 'size'),
    escalation_rate=('stage_f_escalation_event', 'mean'),
    admission_rate=('baseline_admission_event', 'mean'),
    mean_hazard=('hazard_prob_stage_f', 'mean')
)
cycle_stats.to_csv(TABLE_DIR / 'stage_m_cycle_presence_stats.csv', index=False)

plt.figure(figsize=(10, 5))
plt.plot(cycle_stats['cycle_id_stage_f'], cycle_stats['escalation_rate'], marker='o', label='Escalation rate')
plt.plot(cycle_stats['cycle_id_stage_f'], cycle_stats['admission_rate'], marker='s', label='Admission rate')
plt.title('Cycle Presence Justification: Outcome Rates by Cycle')
plt.xlabel('Cycle ID')
plt.ylabel('Rate')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_m_cycle_presence_rates.png', dpi=140, bbox_inches='tight')
plt.close()

plt.figure(figsize=(10, 5))
plt.bar(cycle_stats['cycle_id_stage_f'].astype(str), cycle_stats['mean_hazard'])
plt.title('Mean Hazard by Cycle')
plt.xlabel('Cycle ID')
plt.ylabel('Mean Hazard')
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_m_cycle_hazard_bar.png', dpi=140, bbox_inches='tight')
plt.close()

cycle_stats.head(10)

,cycle_id_stage_f,n,escalation_rate,admission_rate,mean_hazard
0,0,3716831,0.000000,0.000000,0.007046
1,1,1017971,0.042018,0.042018,0.006705
2,2,148323,0.071506,0.000000,0.006252
3,3,15451,0.102065,0.000000,0.005904
4,4,1319,0.127369,0.000000,0.005619
5,5,100,0.160000,0.000000,0.005687
6,6,5,0.400000,0.000000,0.004160


In [4]:
# Present vs prediction-adjusted scenario impact modeling
# Present scenario (reactive): low effective targeting
present_coverage = {'high_complexity': 0.20, 'moderate_complexity': 0.12, 'low_complexity': 0.05}
present_effect = {'high_complexity': 0.10, 'moderate_complexity': 0.07, 'low_complexity': 0.03}

# Prediction-adjusted scenario (proactive): risk-stratified targeting
pred_coverage = {'high_complexity': 0.75, 'moderate_complexity': 0.45, 'low_complexity': 0.15}
pred_effect = {'high_complexity': 0.30, 'moderate_complexity': 0.18, 'low_complexity': 0.08}

impact_base = rec_summary.groupby('patient_category', as_index=False).agg(
    n_patients=('n_patients', 'sum'),
    baseline_escalation=('mean_escalation_rate', 'mean')
)
impact_base['expected_escalations_present'] = impact_base['n_patients'] * impact_base['baseline_escalation'] * (1 - impact_base['patient_category'].map(present_coverage) * impact_base['patient_category'].map(present_effect))
impact_base['expected_escalations_prediction'] = impact_base['n_patients'] * impact_base['baseline_escalation'] * (1 - impact_base['patient_category'].map(pred_coverage) * impact_base['patient_category'].map(pred_effect))
impact_base['escalations_averted_vs_present'] = impact_base['expected_escalations_present'] - impact_base['expected_escalations_prediction']
impact_base['relative_reduction_pct'] = np.where(
    impact_base['expected_escalations_present'] > 0,
    100.0 * impact_base['escalations_averted_vs_present'] / impact_base['expected_escalations_present'],
    np.nan
)
impact_base = impact_base.sort_values('patient_category')
impact_base.to_csv(TABLE_DIR / 'stage_m_present_vs_prediction_impact.csv', index=False)

tot_present = float(impact_base['expected_escalations_present'].sum())
tot_pred = float(impact_base['expected_escalations_prediction'].sum())
tot_averted = tot_present - tot_pred

compare_df = pd.DataFrame({
    'scenario': ['present_reactive', 'prediction_adjusted'],
    'expected_escalations': [tot_present, tot_pred]
})
compare_df.to_csv(TABLE_DIR / 'stage_m_scenario_comparison_totals.csv', index=False)

plt.figure(figsize=(7, 5))
plt.bar(compare_df['scenario'], compare_df['expected_escalations'])
plt.title('Expected Escalations: Present vs Prediction-Adjusted')
plt.ylabel('Expected Escalations')
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_m_present_vs_prediction_bar.png', dpi=140, bbox_inches='tight')
plt.close()

plt.figure(figsize=(8, 5))
plt.bar(impact_base['patient_category'], impact_base['escalations_averted_vs_present'])
plt.title('Escalations Averted by Patient Category')
plt.xlabel('Patient Category')
plt.ylabel('Averted Escalations')
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_m_category_impact_bar.png', dpi=140, bbox_inches='tight')
plt.close()

impact_base

,patient_category,n_patients,baseline_escalation,expected_escalations_present,expected_escalations_prediction,escalations_averted_vs_present,relative_reduction_pct
0,high_complexity,168,0.083887,13.811178,10.922105,2.889073,20.918367
1,low_complexity,57227,0.000000,0.000000,0.000000,0.000000,NaN
2,moderate_complexity,42605,0.025911,1094.668770,1014.522589,80.146181,7.321501


In [5]:
# Sensitivity heatmap: how robust is benefit to coverage/effect assumptions?
coverage_grid = np.linspace(0.20, 0.90, 15)
effect_grid = np.linspace(0.08, 0.40, 15)

base_high = impact_base.loc[impact_base['patient_category'] == 'high_complexity', ['n_patients', 'baseline_escalation']].iloc[0]
high_n = float(base_high['n_patients'])
high_r = float(base_high['baseline_escalation'])

present_high_expected = high_n * high_r * (1 - present_coverage['high_complexity'] * present_effect['high_complexity'])
heat = np.zeros((len(effect_grid), len(coverage_grid)), dtype=float)
for i, eff in enumerate(effect_grid):
    for j, cov in enumerate(coverage_grid):
        pred_expected = high_n * high_r * (1 - cov * eff)
        heat[i, j] = present_high_expected - pred_expected

heat_df = pd.DataFrame(heat, index=[round(x, 3) for x in effect_grid], columns=[round(x, 3) for x in coverage_grid])
heat_df.to_csv(TABLE_DIR / 'stage_m_high_complexity_sensitivity_heatmap.csv', index=True)

plt.figure(figsize=(9, 6))
im = plt.imshow(heat, aspect='auto', origin='lower')
plt.colorbar(im, label='Averted Escalations vs Present (High Complexity)')
plt.xticks(np.arange(len(coverage_grid))[::2], [f'{x:.2f}' for x in coverage_grid[::2]], rotation=45)
plt.yticks(np.arange(len(effect_grid))[::2], [f'{x:.2f}' for x in effect_grid[::2]])
plt.xlabel('Prediction-adjusted intervention coverage')
plt.ylabel('Prediction-adjusted intervention effectiveness')
plt.title('Sensitivity Heatmap: High Complexity Cohort')
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_m_sensitivity_heatmap.png', dpi=140, bbox_inches='tight')
plt.close()

heat_df.head()

,0.20,0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70,0.75,0.80,0.85,0.90
0.080,-0.056372,0.000000,0.056372,0.112744,0.169116,0.225489,0.281861,0.338233,0.394605,0.450977,0.507349,0.563722,0.620094,0.676466,0.732838
0.103,0.008053,0.080532,0.153010,0.225489,0.297967,0.370446,0.442924,0.515403,0.587881,0.660360,0.732838,0.805316,0.877795,0.950273,1.022752
0.126,0.072478,0.161063,0.249648,0.338233,0.426818,0.515403,0.603987,0.692572,0.781157,0.869742,0.958327,1.046911,1.135496,1.224081,1.312666
0.149,0.136904,0.241595,0.346286,0.450977,0.555668,0.660360,0.765051,0.869742,0.974433,1.079124,1.183815,1.288506,1.393198,1.497889,1.602580
0.171,0.201329,0.322127,0.442924,0.563722,0.684519,0.805316,0.926114,1.046911,1.167709,1.288506,1.409304,1.530101,1.650899,1.771696,1.892494


In [6]:
# Resource translation and narrative report
# Assumption knobs (explicit and editable)
admission_share_of_escalation = 0.30
avg_inpatient_bed_days_per_admission = 6.5
case_management_hours_per_high_complexity_patient = 4.0

expected_admissions_present = tot_present * admission_share_of_escalation
expected_admissions_prediction = tot_pred * admission_share_of_escalation
admissions_avoided = expected_admissions_present - expected_admissions_prediction
bed_days_avoided = admissions_avoided * avg_inpatient_bed_days_per_admission

high_n = float(impact_base.loc[impact_base['patient_category'] == 'high_complexity', 'n_patients'].iloc[0])
high_present_covered = high_n * present_coverage['high_complexity']
high_pred_covered = high_n * pred_coverage['high_complexity']
extra_case_mgmt_hours = (high_pred_covered - high_present_covered) * case_management_hours_per_high_complexity_patient

resource_df = pd.DataFrame([
    {'metric': 'expected_escalations_present', 'value': tot_present},
    {'metric': 'expected_escalations_prediction_adjusted', 'value': tot_pred},
    {'metric': 'escalations_averted', 'value': tot_averted},
    {'metric': 'expected_admissions_present', 'value': expected_admissions_present},
    {'metric': 'expected_admissions_prediction_adjusted', 'value': expected_admissions_prediction},
    {'metric': 'admissions_avoided', 'value': admissions_avoided},
    {'metric': 'bed_days_avoided', 'value': bed_days_avoided},
    {'metric': 'additional_case_management_hours', 'value': extra_case_mgmt_hours}
])
resource_df.to_csv(TABLE_DIR / 'stage_m_resource_translation.csv', index=False)

with open(REPORT_DIR / 'stage_m_core_validation_summary.txt', 'w', encoding='utf-8') as f:
    f.write('Stage M Core Validation Summary\n')
    f.write(f'patients: {overall["n_patients"]}\n')
    f.write(f'panel escalation rate: {overall["escalation_rate"]:.6f}\n')
    f.write(f'present expected escalations: {tot_present:.3f}\n')
    f.write(f'prediction-adjusted expected escalations: {tot_pred:.3f}\n')
    f.write(f'escalations averted: {tot_averted:.3f}\n')
    f.write(f'admissions avoided: {admissions_avoided:.3f}\n')
    f.write(f'bed-days avoided: {bed_days_avoided:.3f}\n')
    f.write(f'additional case-management hours: {extra_case_mgmt_hours:.3f}\n')
    f.write('\nInterpretation:\n')
    f.write('- Prediction-adjusted targeting reduces expected escalations under explicit assumptions.\n')
    f.write('- Benefit concentrates in high/moderate complexity cohorts and depends on cycle-informed intervention timing.\n')
    f.write('- Operational gain is a trade: more proactive care hours in exchange for fewer downstream admissions.\n')

manifest_m = {
    'phase': 'M',
    'notebook': '10_stage_m_core_validation_storytelling.ipynb',
    'inputs': [
        'Results/tables/notebook03_phase_f/phase_f_closed_loop_panel.parquet',
        'Results/tables/notebook05_stage_h/stage_h_baseline_model_performance.csv',
        'Results/tables/notebook09_stage_l/stage_l_model_metrics.csv',
        'Results/tables/notebook09_stage_l/stage_l_patient_category_recommendations.csv',
        'Results/tables/notebook09_stage_l/stage_l_medication_recommendation_summary.csv'
    ],
    'outputs_tables': [
        'Results/tables/notebook10_stage_m/stage_m_cycle_presence_stats.csv',
        'Results/tables/notebook10_stage_m/stage_m_present_vs_prediction_impact.csv',
        'Results/tables/notebook10_stage_m/stage_m_scenario_comparison_totals.csv',
        'Results/tables/notebook10_stage_m/stage_m_high_complexity_sensitivity_heatmap.csv',
        'Results/tables/notebook10_stage_m/stage_m_resource_translation.csv'
    ],
    'outputs_figures': [
        'Results/figures/notebook10_stage_m/stage_m_cycle_presence_rates.png',
        'Results/figures/notebook10_stage_m/stage_m_cycle_hazard_bar.png',
        'Results/figures/notebook10_stage_m/stage_m_present_vs_prediction_bar.png',
        'Results/figures/notebook10_stage_m/stage_m_category_impact_bar.png',
        'Results/figures/notebook10_stage_m/stage_m_sensitivity_heatmap.png'
    ],
    'outputs_reports': [
        'Results/reports/notebook10_stage_m/stage_m_core_validation_summary.txt'
    ]
}
with open(META_DIR / 'phase_m_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest_m, f, indent=4)

proof_m = {
    'cycle_presence_generated': (TABLE_DIR / 'stage_m_cycle_presence_stats.csv').exists(),
    'scenario_impact_generated': (TABLE_DIR / 'stage_m_present_vs_prediction_impact.csv').exists(),
    'sensitivity_heatmap_generated': (TABLE_DIR / 'stage_m_high_complexity_sensitivity_heatmap.csv').exists(),
    'resource_translation_generated': (TABLE_DIR / 'stage_m_resource_translation.csv').exists(),
    'manifest_generated': (META_DIR / 'phase_m_manifest.json').exists()
}
with open(REPORT_DIR / 'stage_m_checklist_proof.json', 'w', encoding='utf-8') as f:
    json.dump({'proof': proof_m}, f, indent=4)

print('Stage M validation artifacts generated')
resource_df

Stage M validation artifacts generated


,metric,value
0,expected_escalations_present,1108.479948
1,expected_escalations_prediction_adjusted,1025.444694
2,escalations_averted,83.035254
3,expected_admissions_present,332.543984
4,expected_admissions_prediction_adjusted,307.633408
5,admissions_avoided,24.910576
6,bed_days_avoided,161.918744
7,additional_case_management_hours,369.600000


In [7]:
# Medication label audit (exact labels used in enriched upstream data)
med_audit_path = PROJECT_ROOT / 'Data' / 'medication_event.parquet'
med_audit = pd.read_parquet(med_audit_path)

required_cols = ['patient_id', 'event_day', 'diagnosis_code', 'medication_class', 'medication_name', 'regimen_phase', 'strategy_code', 'enrichment_version']
avail_cols = [c for c in required_cols if c in med_audit.columns]

med_name_counts = med_audit['medication_name'].value_counts().rename_axis('medication_name').reset_index(name='n') if 'medication_name' in med_audit.columns else pd.DataFrame()
med_class_name = med_audit.groupby(['medication_class', 'medication_name']).size().rename('n').reset_index().sort_values(['medication_class', 'n'], ascending=[True, False]) if {'medication_class', 'medication_name'}.issubset(med_audit.columns) else pd.DataFrame()

if len(med_name_counts) > 0:
    med_name_counts.to_csv(TABLE_DIR / 'stage_m_medication_name_counts.csv', index=False)
if len(med_class_name) > 0:
    med_class_name.to_csv(TABLE_DIR / 'stage_m_medication_class_name_counts.csv', index=False)

print('Medication audit columns available:', avail_cols)
print('Unique medication names:', sorted(med_audit['medication_name'].dropna().unique().tolist()) if 'medication_name' in med_audit.columns else [])
print('n unique medication names:', med_audit['medication_name'].nunique() if 'medication_name' in med_audit.columns else 0)
med_audit[avail_cols].head(10)

Medication audit columns available: ['patient_id', 'event_day', 'diagnosis_code', 'medication_class', 'medication_name', 'regimen_phase', 'strategy_code', 'enrichment_version']
Unique medication names: ['aripiprazole_low', 'desvenlafaxine', 'duloxetine', 'escitalopram', 'fluoxetine', 'lamotrigine', 'lithium', 'olanzapine', 'olanzapine_low', 'paroxetine', 'quetiapine', 'quetiapine_low', 'risperidone', 'sertraline', 'valproate', 'venlafaxine']
n unique medication names: 16


,patient_id,event_day,diagnosis_code,medication_class,medication_name,regimen_phase,strategy_code,enrichment_version
0,P000048,0,F32,ssri,sertraline,induction,continue_and_reassess,notebook01_med_enrichment_v1
1,P000209,0,F32,ssri,sertraline,induction,continue_and_reassess,notebook01_med_enrichment_v1
2,P000233,0,F32,ssri,escitalopram,induction,intensify_or_switch,notebook01_med_enrichment_v1
3,P000970,0,F32,ssri,fluoxetine,induction,continue_and_reassess,notebook01_med_enrichment_v1
4,P001092,0,F32,ssri,sertraline,induction,continue_and_reassess,notebook01_med_enrichment_v1
5,P001163,0,F32,ssri,escitalopram,induction,continue_and_reassess,notebook01_med_enrichment_v1
6,P001366,0,F32,ssri,escitalopram,induction,continue_and_reassess,notebook01_med_enrichment_v1
7,P001735,0,F32,ssri,sertraline,induction,continue_and_reassess,notebook01_med_enrichment_v1
8,P002409,0,F32,ssri,fluoxetine,induction,continue_and_reassess,notebook01_med_enrichment_v1
9,P002459,0,F32,ssri,sertraline,induction,continue_and_reassess,notebook01_med_enrichment_v1


In [ ]:
# Inline artifact gallery for this notebook stage
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = PROJECT_ROOT if 'PROJECT_ROOT' in globals() else (Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd())
STAGE_PREFIX = 'notebook10'

def _match_stage_dirs(base, prefix):
    if not base.exists():
        return []
    return sorted([p for p in base.glob(f'{prefix}*') if p.is_dir()])

def _show_table_file(path):
    suffix = path.suffix.lower()
    display(Markdown(f'**{path.name}**'))
    try:
        if suffix == '.csv':
            display(pd.read_csv(path).head(200))
        elif suffix == '.parquet':
            display(pd.read_parquet(path).head(200))
        elif suffix == '.json':
            data = json.loads(path.read_text(encoding='utf-8'))
            if isinstance(data, list):
                display(pd.DataFrame(data).head(200))
            elif isinstance(data, dict):
                display(pd.DataFrame([data]).T.head(200))
            else:
                print(str(data)[:12000])
        elif suffix in {'.txt', '.md'}:
            print(path.read_text(encoding='utf-8')[:12000])
    except Exception as exc:
        print(f'Could not render {path.name}: {exc}')

table_dirs = _match_stage_dirs(ROOT / 'Results' / 'tables', STAGE_PREFIX)
figure_dirs = _match_stage_dirs(ROOT / 'Results' / 'figures', STAGE_PREFIX)
report_dirs = _match_stage_dirs(ROOT / 'Results' / 'reports', STAGE_PREFIX)

display(Markdown(f'## Inline Artifact Gallery: {STAGE_PREFIX}'))
if not table_dirs and not figure_dirs and not report_dirs:
    print('No stage-matched artifact folders found yet. Run generation cells first.')

for d in table_dirs:
    display(Markdown(f'### Tables ({d.name})'))
    files = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.csv', '.parquet', '.json', '.txt'}])
    if not files:
        print('No table files found')
    for fp in files:
        _show_table_file(fp)

for d in figure_dirs:
    display(Markdown(f'### Visualizations ({d.name})'))
    imgs = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.png', '.jpg', '.jpeg'}])
    if not imgs:
        print('No figure files found')
    for fp in imgs:
        display(Markdown(f'**{fp.name}**'))
        display(Image(filename=str(fp)))

for d in report_dirs:
    display(Markdown(f'### Reports ({d.name})'))
    files = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.txt', '.md'}])
    if not files:
        print('No report files found')
    for fp in files:
        _show_table_file(fp)